In [ ]:
# ============================================================
# GPU-optimized training (single process dataloader)
# with ambient noise augmentation (SNR 5-15 dB)
# ============================================================
!pip install evaluate jiwer -q

import os
import torch
import numpy as np
import pandas as pd
import random
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torchaudio
import torchaudio.functional as F
from torch.utils.data import Dataset as TorchDataset
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Paths
AUDIO_BASE = "/kaggle/input/datasets/mosush/audio-files-cpmpresd/audio files - cpmpresd"
CSV_PATH = "/kaggle/input/datasets/mosush/dysphonic-50/dysphonic_50.csv"
ORIGINAL_MODEL_WEIGHTS = "/kaggle/input/datasets/mosush/finale/final/model.safetensors"
BASE_MODEL_NAME = "Harveenchadha/vakyansh-wav2vec2-kannada-knm-560"

# --- NEW: Ambient noise directory (create or point to your noise files) ---
NOISE_DIR = "/kaggle/input/ambient-noise-library"   # contains .wav files (cafeteria, fan, traffic, etc.)

# Load CSV and fix paths
df = pd.read_csv(CSV_PATH)
def fix_path(p):
    p = p.replace("\\", "/")
    parts = p.split("audio files - cpmpresd")
    if len(parts) >= 2:
        rel = parts[-1].lstrip("/")
        return os.path.join(AUDIO_BASE, rel)
    return p
df["path"] = df["path"].apply(fix_path)
print(f"Loaded {len(df)} real dysphonic samples")

# Processor
processor = Wav2Vec2Processor.from_pretrained(BASE_MODEL_NAME)

# ------------------------------------------------------------
# Precompute input_values and labels for all original samples
# ------------------------------------------------------------
print("Precomputing input values for original samples...")
original_inputs = []
original_labels = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    audio, sr = torchaudio.load(row["path"])
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        audio = resampler(audio)
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)
    audio = audio.squeeze(0).numpy()
    max_val = np.abs(audio).max()
    if max_val > 1.0:
        audio = audio / max_val
    input_values = processor(audio, sampling_rate=16000).input_values[0]
    labels = processor.tokenizer(row["transcript"], truncation=True, max_length=128).input_ids
    original_inputs.append(torch.tensor(input_values, dtype=torch.float32))
    original_labels.append(torch.tensor(labels, dtype=torch.long))

print(f"Precomputed {len(original_inputs)} samples")

# ------------------------------------------------------------
# Load ambient noise library (pre‑resampled to 16 kHz)
# ------------------------------------------------------------
noise_files = [os.path.join(NOISE_DIR, f) for f in os.listdir(NOISE_DIR) if f.endswith('.wav')]
noise_list = []
for nf in noise_files:
    noise, sr_n = torchaudio.load(nf)
    if sr_n != 16000:
        resampler = torchaudio.transforms.Resample(sr_n, 16000)
        noise = resampler(noise)
    if noise.shape[0] > 1:
        noise = torch.mean(noise, dim=0, keepdim=True)
    noise_list.append(noise.squeeze(0))   # shape (samples,)
print(f"Loaded {len(noise_list)} ambient noise clips")

# ------------------------------------------------------------
# Augmentation with SNR‑controlled ambient noise
# ------------------------------------------------------------
def add_ambient_noise_with_snr(waveform, sample_rate=16000, snr_db_range=(5, 15)):
    """
    waveform: 1D torch tensor, values in [-1, 1]
    returns waveform with added noise at randomly chosen SNR within snr_db_range
    """
    if len(noise_list) == 0:
        return waveform
    # Pick random noise clip
    noise = random.choice(noise_list).clone()
    # Trim or pad noise to match waveform length
    if noise.shape[0] > waveform.shape[0]:
        start = random.randint(0, noise.shape[0] - waveform.shape[0])
        noise = noise[start:start + waveform.shape[0]]
    elif noise.shape[0] < waveform.shape[0]:
        repeats = (waveform.shape[0] // noise.shape[0]) + 1
        noise = noise.repeat(repeats)[:waveform.shape[0]]
    # Compute RMS of speech and noise
    rms_speech = torch.sqrt(torch.mean(waveform ** 2) + 1e-12)
    rms_noise = torch.sqrt(torch.mean(noise ** 2) + 1e-12)
    # Target SNR dB (random between 5 and 15)
    target_snr_db = random.uniform(snr_db_range[0], snr_db_range[1])
    # Scale noise: desired_rms_noise = rms_speech / (10^(SNR/20))
    desired_rms_noise = rms_speech / (10 ** (target_snr_db / 20))
    scaling_factor = desired_rms_noise / (rms_noise + 1e-12)
    noise_scaled = noise * scaling_factor
    return waveform + noise_scaled

def augment_input_values_cpu(input_tensor, sample_rate=16000):
    """Apply gain, pitch shift, and SNR‑controlled ambient noise."""
    waveform = input_tensor.clone()
    # Gain
    if random.random() < 0.8:
        gain = random.uniform(0.6, 1.4)
        waveform = waveform * gain
    # Pitch shift
    if random.random() < 0.7:
        n_steps = random.uniform(-1.5, 1.5)
        waveform = waveform.unsqueeze(0)
        waveform = F.pitch_shift(waveform, sample_rate, n_steps)
        waveform = waveform.squeeze(0)
    # Ambient noise with SNR 5-15 dB (replaces simple Gaussian)
    if random.random() < 0.6:          # 60% probability to add noise
        waveform = add_ambient_noise_with_snr(waveform, sample_rate, snr_db_range=(5, 15))
    return torch.clamp(waveform, -1.0, 1.0)

# ------------------------------------------------------------
# Custom Dataset
# ------------------------------------------------------------
class FastAugmentedDataset(TorchDataset):
    def __init__(self, input_tensors, label_tensors, is_train=True):
        self.input_tensors = input_tensors
        self.label_tensors = label_tensors
        self.is_train = is_train
    
    def __len__(self):
        if self.is_train:
            return len(self.input_tensors) * 20
        else:
            return len(self.input_tensors)
    
    def __getitem__(self, idx):
        if self.is_train:
            orig_idx = idx // 20
            input_vals = self.input_tensors[orig_idx].clone()
            input_vals = augment_input_values_cpu(input_vals)
        else:
            orig_idx = idx
            input_vals = self.input_tensors[orig_idx].clone()
        return {"input_values": input_vals, "labels": self.label_tensors[orig_idx].clone()}

# Split indices
indices = list(range(len(df)))
random.seed(42)
random.shuffle(indices)
split_point = int(0.9 * len(indices))
train_indices = indices[:split_point]
eval_indices = indices[split_point:]

train_inputs = [original_inputs[i] for i in train_indices]
train_labels = [original_labels[i] for i in train_indices]
eval_inputs = [original_inputs[i] for i in eval_indices]
eval_labels = [original_labels[i] for i in eval_indices]

print(f"Train original samples: {len(train_inputs)} -> will generate {len(train_inputs)*20} per epoch")
print(f"Eval samples: {len(eval_inputs)}")

train_dataset = FastAugmentedDataset(train_inputs, train_labels, is_train=True)
eval_dataset = FastAugmentedDataset(eval_inputs, eval_labels, is_train=False)

# ------------------------------------------------------------
# Data collator
# ------------------------------------------------------------
from dataclasses import dataclass
from typing import Dict, List, Union
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features: List[Dict]) -> Dict:
        input_features = [{"input_values": f["input_values"].numpy()} for f in features]
        label_features = [{"input_ids": f["labels"].numpy()} for f in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        if "attention_mask" not in batch:
            batch["attention_mask"] = (batch["input_values"] != 0.0).long()
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
import evaluate
wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

# ------------------------------------------------------------
# Model loading
# ------------------------------------------------------------
print("Loading base model architecture...")
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_NAME,
    gradient_checkpointing=False,
    pad_token_id=processor.tokenizer.pad_token_id,
)
print("Loading original fine-tuned weights...")
from safetensors.torch import load_file
state_dict = load_file(ORIGINAL_MODEL_WEIGHTS)
model.load_state_dict(state_dict, strict=True)
model.config.ctc_loss_reduction = "mean"
model.config.dropout = 0.4
model.config.feat_proj_dropout = 0.4
model.config.attention_dropout = 0.4
model.config.hidden_dropout = 0.4
model = model.to("cuda")
model.train()

# ------------------------------------------------------------
# Training arguments (single worker)
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="/kaggle/working/original_on_real_20aug",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    learning_rate=5e-5,
    warmup_steps=100,
    num_train_epochs=15,
    weight_decay=0.03,
    max_grad_norm=1.0,
    fp16=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=3,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting training with GPU-optimized pipeline (single worker) and ambient noise augmentation (SNR 5–15 dB)...")
trainer.train()

eval_results = trainer.evaluate()
print(f"Final loss: {eval_results['eval_loss']:.4f}, Final WER: {eval_results['eval_wer']:.4f}")

final_dir = "/kaggle/working/original_model_20aug_gpu"
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
!zip -r /kaggle/working/original_model_20aug_gpu.zip /kaggle/working/original_model_20aug_gpu
print("Model saved to original_model_20aug_gpu.zip")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.3 MB/s eta 0:00:00a 0:00:01
Loaded 50 real dysphonic samples


preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/adapted_model_perfect")

In [ ]:
import os
os.remove("/kaggle/working/adapted_model_perfect.zip")